# 02 — Silver
Cleans bronze into tables that are safe to aggregate.

Every rule below is a decision, not a default. Each one is measured against bronze first,
printed, and only then applied — so the README can say how many rows each rule touched
instead of claiming the data was "cleaned".

The one rule that is a decision *not* to act is deduplication. See `Duplicates` below.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

bronze_tx = spark.table("bronze_transactions")
bronze_ar = spark.table("bronze_articles")

## Profile transactions
One pass over 31.8M rows computing every number the rules below depend on. Done as a
single `agg` rather than one `count` per condition, so Spark scans the table once.

In [0]:
parsed_date = F.to_date("t_dat", "yyyy-MM-dd")

tx_profile = bronze_tx.agg(
    F.count(F.lit(1)).alias("rows"),
    F.sum(F.when(F.col("t_dat").isNull(), 1).otherwise(0)).alias("null_date"),
    F.sum(F.when(F.col("t_dat").isNotNull() & parsed_date.isNull(), 1).otherwise(0)).alias("unparseable_date"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer"),
    F.sum(F.when(F.col("article_id").isNull(), 1).otherwise(0)).alias("null_article"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("price") <= 0, 1).otherwise(0)).alias("non_positive_price"),
    F.sum(F.when(~F.col("sales_channel_id").isin(1, 2), 1).otherwise(0)).alias("unknown_channel"),
    F.min(parsed_date).alias("first_date"),
    F.max(parsed_date).alias("last_date"),
).collect()[0]

for field in tx_profile.asDict():
    print(f"{field:>20}: {tx_profile[field]}")

                rows: 31788324
           null_date: 0
    unparseable_date: 0
       null_customer: 0
        null_article: 0
          null_price: 0
  non_positive_price: 0
     unknown_channel: 0
          first_date: 2018-09-20
           last_date: 2020-09-22


## Duplicates
A customer buying two of the same article on the same day produces two byte-identical
rows. The source has no line number, order id or quantity column, so an identical row is
indistinguishable from a real second unit — and dropping it would silently delete sales.

**Decision: keep them.** The number below is the count of rows that would disappear under
a naive `dropDuplicates()`, which is why the rule is written down rather than assumed.

In [0]:
tx_rows = tx_profile["rows"]
tx_distinct = bronze_tx.select(
    "t_dat", "customer_id", "article_id", "price", "sales_channel_id"
).distinct().count()

print(f"rows:              {tx_rows:,}")
print(f"distinct rows:     {tx_distinct:,}")
print(f"would be dropped:  {tx_rows - tx_distinct:,}  ({(tx_rows - tx_distinct) / tx_rows:.2%}) — kept")

rows:              31,788,324
distinct rows:     28,813,419
would be dropped:  2,974,905  (9.36%) — kept


## Cleaning rules — transactions
1. `t_dat` becomes a real `DATE`. Rows whose date is null or does not parse are dropped:
   a sale with no date cannot be aggregated by time, which is what gold does.
2. Rows with a null `customer_id` or `article_id` are dropped — they join to nothing.
3. Rows with a null or non-positive `price` are dropped. The competition data has no
   returns or refunds, so a zero or negative price is corruption, not a credit note.
4. `sales_channel_id` gains a readable `sales_channel` label (1 = store, 2 = online).
   The original id is kept so nothing is lost.
5. Duplicates are kept, per the section above.

`price` is left exactly as it is. It is an H&M-scaled value, **not** a currency amount;
converting it would be inventing a number.

In [0]:
silver_tx = (
    bronze_tx
    .withColumn("t_dat", parsed_date)
    .filter(F.col("t_dat").isNotNull())
    .filter(F.col("customer_id").isNotNull() & F.col("article_id").isNotNull())
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    .withColumn(
        "sales_channel",
        F.when(F.col("sales_channel_id") == 1, "store")
         .when(F.col("sales_channel_id") == 2, "online")
         .otherwise("unknown"),
    )
    .select(
        "t_dat", "customer_id", "article_id", "price",
        "sales_channel_id", "sales_channel",
        "_source_file", "_ingested_at",
    )
)

(silver_tx.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_transactions"))

## Profile articles

In [0]:
ar_rows = bronze_ar.count()
ar_distinct_ids = bronze_ar.select("article_id").distinct().count()

ar_profile = bronze_ar.agg(
    F.sum(F.when(F.col("article_id").isNull(), 1).otherwise(0)).alias("null_article_id"),
    F.sum(F.when(F.col("detail_desc").isNull(), 1).otherwise(0)).alias("null_detail_desc"),
    F.sum(F.when(F.trim(F.col("detail_desc")) == "", 1).otherwise(0)).alias("empty_detail_desc"),
).collect()[0]

print(f"rows:               {ar_rows:,}")
print(f"distinct article_id:{ar_distinct_ids:,}")
print(f"duplicate ids:      {ar_rows - ar_distinct_ids:,}")
for field in ar_profile.asDict():
    print(f"{field:>20}: {ar_profile[field]:,}")

rows:               105,542
distinct article_id:105,542
duplicate ids:      0
     null_article_id: 0
    null_detail_desc: 416
   empty_detail_desc: 0


## Cleaning rules — articles
1. Rows with a null `article_id` are dropped — the product key is the whole point of the table.
2. One row per `article_id`. Unlike transactions, this is a product dimension: a repeated
   key is a defect, not a second sale. The count above says whether any exist.
3. `detail_desc` is trimmed, and an empty string becomes `NULL`. This is the text that
   gets embedded for retrieval later, and `""` is not a description — making it NULL means
   the retrieval step can exclude it honestly instead of indexing blanks.
4. `article_id` stays a STRING. Leading zeros are part of the identifier.

In [0]:
from pyspark.sql.window import Window

dedupe = Window.partitionBy("article_id").orderBy(F.col("_ingested_at").asc())

silver_ar = (
    bronze_ar
    .filter(F.col("article_id").isNotNull())
    .withColumn(
        "detail_desc",
        F.when(F.trim(F.col("detail_desc")) == "", None).otherwise(F.trim(F.col("detail_desc"))),
    )
    .withColumn("_rn", F.row_number().over(dedupe))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

(silver_ar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_articles"))

## Row counts
Bronze in, silver out, and what the rules removed. These numbers go in the README.

In [0]:
counts = {
    "bronze_transactions": spark.table("bronze_transactions").count(),
    "silver_transactions": spark.table("silver_transactions").count(),
    "bronze_articles": spark.table("bronze_articles").count(),
    "silver_articles": spark.table("silver_articles").count(),
}

for table, n in counts.items():
    print(f"{CATALOG}.{SCHEMA}.{table}: {n:,} rows")

print()
for name in ["transactions", "articles"]:
    dropped = counts[f"bronze_{name}"] - counts[f"silver_{name}"]
    pct = dropped / counts[f"bronze_{name}"]
    print(f"{name}: dropped {dropped:,} rows ({pct:.4%})")

workspace.retail.bronze_transactions: 31,788,324 rows
workspace.retail.silver_transactions: 31,788,324 rows
workspace.retail.bronze_articles: 105,542 rows
workspace.retail.silver_articles: 105,542 rows

transactions: dropped 0 rows (0.0000%)
articles: dropped 0 rows (0.0000%)


In [0]:
display(spark.table("silver_transactions").limit(5))

t_dat,customer_id,article_id,price,sales_channel_id,sales_channel,_source_file,_ingested_at
2019-06-20,e55d32c80364ecb0041e711a8d8cada0ffc94565064c03eb4132971cfbe57167,0561797001,0.011847457627118646,2,online,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-06-20,e55d32c80364ecb0041e711a8d8cada0ffc94565064c03eb4132971cfbe57167,0724164002,0.027101694915254236,2,online,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-06-20,e55d32c80364ecb0041e711a8d8cada0ffc94565064c03eb4132971cfbe57167,0252298015,0.020322033898305086,2,online,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-06-20,e55d32c80364ecb0041e711a8d8cada0ffc94565064c03eb4132971cfbe57167,0727804001,0.03049152542372881,2,online,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z
2019-06-20,e55d32c80364ecb0041e711a8d8cada0ffc94565064c03eb4132971cfbe57167,0699178001,0.022016949152542376,1,store,dbfs:/Volumes/workspace/retail/raw/transactions_train.csv,2026-10-13T09:44:11.776Z


In [0]:
display(spark.table("silver_articles").select(
    "article_id", "prod_name", "product_type_name", "colour_group_name", "detail_desc"
).limit(5))

article_id,prod_name,product_type_name,colour_group_name,detail_desc
0108775015,Strap top,Vest top,Black,Jersey top with narrow shoulder straps.
0108775044,Strap top,Vest top,White,Jersey top with narrow shoulder straps.
0108775051,Strap top (1),Vest top,Off White,Jersey top with narrow shoulder straps.
0110065001,OP T-shirt (Idro),Bra,Black,"Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort."
0110065002,OP T-shirt (Idro),Bra,White,"Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort."
